In [1]:
import os
import pandas as pd
from tqdm import tqdm

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.0' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


### Constants

In [2]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

int_threshold = 24 # threshold for target months

# columns to ignore
list_cols_id = [
    'UniqueID',
    'bigAccountId',
    'bigDebtorId',
    'bitDebtor',
    'uniqueid',
]

Project: 20231010-gen-xii


### Import Target Data Frame

In [3]:
%%time

str_filename = 'GenXIIPerformanceMonitoringTarget.csv'
str_uri = f's3://{str_project}/11_monitoring/input/{str_filename}'
df = pd.read_csv(str_uri)

# show
df

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:272: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


CPU times: user 5.19 s, sys: 960 ms, total: 6.15 s
Wall time: 15.8 s


,UniqueID,bigAccountId,bigDebtorId,bitDebtor,Default_1,Default_2,Default_3,Default_4,Default_5,Default_6,...,DQ90_63,DQ90_64,DQ90_65,DQ90_66,DQ90_67,DQ90_68,DQ90_69,DQ90_70,DQ90_71,DQ90_72
0,133754717564901,1337547,1756490,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,133757917565301,1337579,1756530,1,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,1
2,133769917566881,1337699,1756688,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,133786117569021,1337861,1756902,1,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,1
4,133789717569521,1337897,1756952,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196834,481006061269301,4810060,6126931,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
196835,481159061288331,4811590,6128834,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
196836,481194961292901,4811949,6129291,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
196837,481249761299911,4812497,6129992,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Remove rows with duplicate Unique IDs

In [4]:
%%time

df = df.drop_duplicates(
    subset=['UniqueID'],
    keep='last',
)

# save
str_filename = 'df_monitoring_targets.csv'
str_uri = f's3://{str_project}/11_monitoring/input/{str_filename}'
df.to_csv(str_uri, index=False)

# show
df

CPU times: user 10.3 s, sys: 220 ms, total: 10.5 s
Wall time: 14.1 s


,UniqueID,bigAccountId,bigDebtorId,bitDebtor,Default_1,Default_2,Default_3,Default_4,Default_5,Default_6,...,DQ90_63,DQ90_64,DQ90_65,DQ90_66,DQ90_67,DQ90_68,DQ90_69,DQ90_70,DQ90_71,DQ90_72
0,133754717564901,1337547,1756490,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,133786117569021,1337861,1756902,1,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,1
4,133789717569521,1337897,1756952,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,133791117569691,1337911,1756969,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,133792317569821,1337923,1756982,1,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196834,481006061269301,4810060,6126931,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
196835,481159061288331,4811590,6128834,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
196836,481194961292901,4811949,6129291,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
196837,481249761299911,4812497,6129992,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Find columns with no variance in train or valid

In [5]:
list_cols_novar = []
for str_df in tqdm(['train','valid']):
    # import the unique ID from the raw data
    str_filename = f'df_{str_df}_raw.gzip'
    str_uri = f's3://{str_project}/02_pricing_pd/01_data_prep/03_train_valid_test_split/{str_filename}'
    list_cols = [
        'uniqueid',
    ]
    df_tmp = pd.read_parquet(str_uri, columns=list_cols)
    # join with the targets
    df_tmp = pd.merge(
        left=df_tmp,
        right=df,
        left_on='uniqueid',
        right_on='UniqueID',
        how='left',
    )
    # check each target for variance
    list_cols = [col for col in df_tmp.columns if col not in list_cols_id]
    for col in list_cols:
        # get sum
        int_sum = df_tmp[col].sum()
        # logic
        if (int_sum == 0) or (int_sum == df_tmp.shape[0]):
            list_cols_novar.append(col)
        else:
            pass
# rm dups
list_cols_novar = list(dict.fromkeys(list_cols_novar))
print(f'There are {len(list_cols_novar)} columns with no variance:')
print(list_cols_novar)

100%|██████████| 2/2 [00:01<00:00,  1.77it/s]

There are 8 columns with no variance:
['Default_1', 'DQ90_0', 'DQ90_1', 'DQ60_0', 'DQ60_1', 'DQ60_2', 'DQ90_2', 'DQ90_3']


### Get list of targets

In [6]:
# rm ids
list_cols = [col for col in df.columns if col not in list_cols_id]

# rm this without variance
list_cols = [col for col in list_cols if col not in list_cols_novar]

# get all columns where the value is <= int_threshold
list_cols_targets = [col for col in list_cols if int(col.split('_')[1]) <= int_threshold]

# create df
df = pd.DataFrame({
    'Target': list_cols_targets,
})

# save
str_filename = 'df_targets.csv'
str_uri = f's3://{str_project}/11_monitoring/input/{str_filename}'
df.to_csv(str_uri, index=False)

# show
df

,Target
0,Default_2
1,Default_3
2,Default_4
3,Default_5
4,Default_6
...,...
136,DQ90_20
137,DQ90_21
138,DQ90_22
139,DQ90_23
